# Chapter 1 — round 2 · strengthening the result

**Accelerator: GPU T4 ×1** (same as before — `fp16` + `sdpa`, nothing changes).

The paper is written and the headline is in it. This notebook adds only what
makes it harder to attack. Nothing here retrains a Chapter 1 condition.

### What we already have (in `results/`, already in the paper)

| | MRR | vs chance 0.0900 |
|---|---|---|
| A real names | **0.8169** | +0.7269 |
| S names permuted | 0.2974 | +0.2074 |
| B anonymised | 0.1336 | +0.0436 |
| C anon + types | 0.0945 | **+0.0045 — at chance** |

→ memorisation share **94.0 %**, split 71.5 % binding / 22.5 % readability / 6.0 % residual.

### The three weaknesses a referee will name, in order

1. **One graph.** The decomposition exists only on YAGO3-10.
2. **One seed, no intervals.** Nothing has a confidence interval.
3. **One model, 1.5B.** *Cannot be fixed on a T4 — we narrow the claim instead.*

### ⚠️ What is gone, and why we are not rebuilding it

The YAGO3-10 adapters and data did not survive the session. **We are not
retraining them.** Every number they produced is already saved in `results/`
and already in the paper. Retraining D and G to add two optional rows would cost
~3 GPU-h; the runs below cost ~1 GPU-h and fix weaknesses 1 and 2, which matter
far more.

### What survived, and it is exactly what we need

`checkpoints/ch1-WN11-lora` and `ch1-WN11-anon-lora` — verified: 112 LoRA
tensors, `q_proj`/`v_proj`, r=8 α=16, Qwen2.5-1.5B-Instruct.

★ These are an **independent run** of WN11 conditions A and B (train loss 0.04476
/ 0.11734, against today's 0.04592 / 0.11784). So they give us **a second graph
AND a second seed at once**, with zero training.

## 0 · Setup

In [ ]:
REPO_URL = 'https://github.com/lynda-lagh/contribution-.git'
DEST     = '/kaggle/working/repo'
LIMIT    = 500          # ranking queries. 500 x 50 = 25k passes ~ 20 min on a T4
N_WAY    = 50

import os, sys, json, glob, time, socket, subprocess
from pathlib import Path

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('No network. Settings -> Internet -> ON.')

def sh(*cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(f'{DEST}/.git'):
    sh('git','-C',DEST,'fetch','--depth','1','origin','main')
    sh('git','-C',DEST,'reset','--hard','FETCH_HEAD')
else:
    sh('git','clone','--depth','1',REPO_URL,DEST)
os.chdir(DEST); sys.path.insert(0, DEST)
print('repo', sh('git','-C',DEST,'log','-1','--pretty=%h %s'))

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('gpu:', gpu)
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x1'

### 0b · restore the two surviving adapters

They are **not** in the repo (too large / not committed). Upload them **as a
Kaggle Dataset**, not to `/kaggle/working` — that is precisely what was lost.

1. On your computer, zip `checkpoints/ch1-WN11-lora` and
   `checkpoints/ch1-WN11-anon-lora` (~4.2 MB each).
2. Kaggle → *Datasets* → *New Dataset* → upload → name it `ch1-wn11-adapters`.
3. In this notebook → *Add Input* → your dataset.

The cell below finds them wherever they landed and verifies they load.

In [ ]:
# locate the adapters, wherever they were mounted
import glob, shutil
from pathlib import Path

WANT = {'real': 'ch1-WN11-lora', 'anon': 'ch1-WN11-anon-lora'}
FOUND = {}

for key, name in WANT.items():
    hits = [p for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True)
            if Path(p, 'adapter_config.json').exists()]
    hits += [p for p in glob.glob(f'checkpoints/{name}')
             if Path(p, 'adapter_config.json').exists()]
    if hits:
        FOUND[key] = hits[0]

print(f'{"arm":6s} {"path":58s} status')
for key, name in WANT.items():
    p = FOUND.get(key)
    if not p:
        print(f'{key:6s} {"—":58s} ✋ NOT FOUND')
        continue
    cfg = json.loads(Path(p, 'adapter_config.json').read_text())
    ok = (cfg.get('r') == 8 and cfg.get('lora_alpha') == 16
          and sorted(cfg.get('target_modules', [])) == ['q_proj','v_proj'])
    print(f'{key:6s} {p[:58]:58s} {"✓ r=8 a=16 q,v" if ok else "⚠️ CONFIG MISMATCH"}')
    print(f'{"":6s} base: {cfg.get("base_model_name_or_path")}')

if len(FOUND) < 2:
    raise SystemExit(
        '✋ adapters not found. Add them as a Kaggle Dataset (see the cell above).\n'
        '   Without them this notebook can only run the untuned rows.')

# ⚠️ copy to the names rank.py expects, so --condition stays meaningful
Path('checkpoints').mkdir(exist_ok=True)
for key, cond in (('real','A'), ('anon','B')):
    dst = Path('checkpoints', f'ch1-WN11-{cond}-run2')
    if not dst.exists():
        shutil.copytree(FOUND[key], dst)
    print('->', dst)

### 0c · WN11 data

WN11 base data survived in the repo (`entity2text.txt`, `relation2text.txt`,
`train.tsv`, `test.tsv`, 21 088 test triples, exactly balanced).

⚠️ These adapters predate the type-consistent negative regeneration. That does
**not** affect ranking — `rank.py` samples its own 50-way candidates and never
reads the negative labels in `test.tsv`. It would affect classification, which
is why we do not re-run classification here.

In [ ]:
!python -m chapter1.validate --dataset WN11 || echo '(validate reported issues — read them)'

from src.data.loaders import load_kg
kg = load_kg('WN11', 'data')
pos = sum(1 for t in kg.test if t.label == 1)
print(f'\nWN11: {len(kg.ent2txt):,} entities · {len(kg.rel2txt)} relations')
print(f'      {len(kg.train):,} train · {len(kg.test):,} test ({pos:,} positive)')
print(f'      ranking uses the {pos:,} POSITIVE triples as queries')

## 1 · ★★ WN11 ranking — the second graph, ~40 min

This is the cell that matters. Four runs, **no training**:

| run | what it buys |
|---|---|
| A tuned | the WN11 baseline |
| B tuned | the WN11 anonymised arm → **memorisation share on a 2nd graph** |
| A untuned | ★ separates *pretraining* from *fine-tuning* |
| B untuned | the anonymised floor without tuning |

★ **The untuned rows are the answer to the scale objection.** A referee will say
"1.5B is too small to do anything but memorise." If the *untuned* model already
ranks far above chance on real names, the knowledge is in the pretrained
backbone, and that is a statement about the base model rather than about our
fine-tuning recipe. It generalises upward far more credibly.

In [ ]:
import subprocess, time
from pathlib import Path

def rank(cond, adapter=None, tag=None, dataset='WN11', limit=LIMIT):
    tag = tag or f'ch1rank-{dataset}-{cond}-P0'
    out = Path('results', f'{tag}.json')
    if out.exists():
        print(f'[skip] {tag} exists'); return
    cmd = ['python','-m','chapter1.rank','--dataset',dataset,'--condition',cond,
           '--limit',str(limit),'--n-way',str(N_WAY),'--tag',tag]
    if adapter:
        cmd += ['--adapter', adapter]
    print('$', ' '.join(cmd), flush=True)
    t0 = time.time()
    rc = subprocess.call(cmd)
    print(f'   rc={rc}   {(time.time()-t0)/60:.1f} min\n', flush=True)

# tuned — the surviving run-2 adapters
rank('A', 'checkpoints/ch1-WN11-A-run2', 'ch1rank-WN11-A-run2')
rank('B', 'checkpoints/ch1-WN11-B-run2', 'ch1rank-WN11-B-run2')

# ★ untuned — no adapter at all
rank('A', None, 'ch1rank-WN11-A-untuned')
rank('B', None, 'ch1rank-WN11-B-untuned')

## 2 · ★ confidence intervals — free, no GPU

Weakness 2. The paper currently reports point estimates with no dispersion.

We reuse `chapter3/stats.py` unchanged: the **paired bootstrap** resamples the
per-query difference, so query difficulty cancels and the interval is roughly
half the unpaired width. A difference is a result when its 95 % interval
**excludes zero**.

⚠️ Requires per-query ranks in the rank JSONs. If a file has none, the cell says
so and falls back to the unpaired interval rather than inventing one.

In [ ]:
import json, glob, math
from pathlib import Path
from chapter3.stats import bootstrap_ci, paired_bootstrap, fmt_ci, fmt_diff, verdict

CH = sum(1/k for k in range(1, N_WAY+1)) / N_WAY

def load(tag):
    p = Path('results', f'{tag}.json')
    if not p.exists(): return None
    return json.loads(p.read_text())

def ranks_of(d):
    for key in ('ranks','per_query','rows'):
        v = d.get(key)
        if isinstance(v, list) and v:
            if isinstance(v[0], (int, float)): return list(v)
            if isinstance(v[0], dict) and 'rank' in v[0]: return [r['rank'] for r in v]
    return None

TAGS = {
    'YAGO A (real)':    'ch1rank-YAGO3-10-A-P0',
    'YAGO S (permuted)':'ch1rank-YAGO3-10-S-P0',
    'YAGO B (anon)':    'ch1rank-YAGO3-10-B-P0',
    'YAGO C (anon+typ)':'ch1rank-YAGO3-10-C-P0',
    'WN11 A (real)':    'ch1rank-WN11-A-run2',
    'WN11 B (anon)':    'ch1rank-WN11-B-run2',
    'WN11 A untuned':   'ch1rank-WN11-A-untuned',
    'WN11 B untuned':   'ch1rank-WN11-B-untuned',
}

print(f'chance MRR = {CH:.4f}\n')
print(f'{"condition":20s} {"MRR":>8s}  {"95% CI":>22s}  n')
print('-'*62)
R = {}
for name, tag in TAGS.items():
    d = load(tag)
    if not d:
        print(f'{name:20s} {"—":>8s}  (not run)'); continue
    m = d['metrics']; rk = ranks_of(d)
    R[name] = (m, rk)
    if rk:
        ci = bootstrap_ci([1/r for r in rk])
        print(f'{name:20s} {m["MRR"]:>8.4f}  [{ci["lo"]:.4f}, {ci["hi"]:.4f}]   {len(rk)}')
    else:
        print(f'{name:20s} {m["MRR"]:>8.4f}  {"(no per-query ranks)":>22s}')

print('\n' + '='*62)
print('PAIRED COMPARISONS — the decomposition, with intervals')
print('='*62)
for a, b in [('YAGO A (real)','YAGO S (permuted)'),
             ('YAGO S (permuted)','YAGO B (anon)'),
             ('YAGO A (real)','YAGO B (anon)'),
             ('YAGO B (anon)','YAGO C (anon+typ)'),
             ('WN11 A (real)','WN11 B (anon)'),
             ('WN11 A (real)','WN11 A untuned')]:
    if a not in R or b not in R: continue
    ra, rb = R[a][1], R[b][1]
    if not (ra and rb and len(ra) == len(rb)):
        print(f'{a} vs {b}: cannot pair (missing or misaligned ranks)'); continue
    t = paired_bootstrap([1/x for x in ra], [1/x for x in rb])
    print(f'\n{a}  −  {b}')
    print(f'   {fmt_diff(t)}')
    print(f'   {verdict(t, a, b)}')

## 3 · the paper table — both graphs side by side

Paste the LaTeX straight into `main.tex`. If the WN11 memorisation share lands
near YAGO3-10's **94.0 %**, the claim stops being a property of one graph.

In [ ]:
def share(A, B):
    return (A - B) / (A - CH)

print(f'{"":22s} {"MRR":>8s} {"H@1":>7s} {"H@10":>7s}')
rows = []
for name in TAGS:
    if name not in R: continue
    m = R[name][0]
    print(f'{name:22s} {m["MRR"]:>8.4f} {m["hits@1"]:>7.3f} {m["hits@10"]:>7.3f}')

print('\n' + '='*62)
print('MEMORISATION SHARE BY GRAPH')
print('='*62)
for g, (a, b) in {'YAGO3-10': ('YAGO A (real)','YAGO B (anon)'),
                  'WN11':     ('WN11 A (real)','WN11 B (anon)')}.items():
    if a in R and b in R:
        A, B = R[a][0]['MRR'], R[b][0]['MRR']
        print(f'  {g:10s}  A={A:.4f}  B={B:.4f}  → {share(A,B):.1%}')

if 'WN11 A untuned' in R and 'WN11 A (real)' in R:
    u, t = R['WN11 A untuned'][0]['MRR'], R['WN11 A (real)'][0]['MRR']
    print(f'\n  ★ WN11 untuned MRR {u:.4f} vs tuned {t:.4f}')
    print(f'    pretraining supplies {(u-CH)/(t-CH):.1%} of the tuned skill')
    print('    → the higher this is, the more the finding is about the BASE model,')
    print('      which is the answer to "1.5B is too small to generalise".')

# ---- LaTeX, ready to paste -------------------------------------------------
print('\n' + '='*62 + '\nLATEX\n' + '='*62)
print(r'\begin{tabular}{@{}llrrr@{}}')
print(r'\toprule')
print(r'\textbf{Graph} & \textbf{Cond.} & \textbf{MRR} & \textbf{H@1} & '
      r'\textbf{$-$chance} \\')
print(r'\midrule')
for g, keys in {'YAGO3-10': ['YAGO A (real)','YAGO S (permuted)','YAGO B (anon)','YAGO C (anon+typ)'],
                'WN11':     ['WN11 A (real)','WN11 B (anon)']}.items():
    for i, k in enumerate(keys):
        if k not in R: continue
        m = R[k][0]
        lbl = k.split('(')[-1].rstrip(')') if '(' in k else k.split()[-1]
        print(f'{g if i==0 else "":9s} & {lbl:12s} & {m["MRR"]:.4f} & '
              f'{m["hits@1"]:.3f} & ${m["MRR"]-CH:+.4f}$ \\\\')
    print(r'\midrule')
print(f'\\multicolumn{{2}}{{@{{}}l}}{{chance}} & {CH:.4f} & 0.020 & --- \\\\')
print(r'\bottomrule')
print(r'\end{tabular}')

## 4 · ✋ BACK UP BEFORE THE SESSION ENDS

This is why the YAGO3-10 adapters are gone. `/kaggle/working` does **not**
persist. Run this, then **download the zip**, every session.

In [ ]:
import datetime, subprocess
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
out = f'/kaggle/working/ch1_round2_{stamp}.zip'

subprocess.call(f'zip -qr {out} results/', shell=True)
subprocess.call('python -m scripts.export_adapters --zip --prune', shell=True)

print('wrote', out)
subprocess.call(f'ls -la {out}', shell=True)
print('\n✋ DOWNLOAD IT NOW — right panel -> Output -> download.')
print('   And push adapters to a Kaggle DATASET, not /kaggle/working.')

## What this notebook does *not* do

| | why not |
|---|---|
| retrain YAGO3-10 A–S | ~3 GPU-h to reproduce numbers already saved in `results/` and already written into the paper |
| rank D and G | needs those adapters. Two optional rows; the two-graph result is worth more |
| re-run classification | three conditions are degenerate (E collapsed to always-No, TPR 0.002). More seeds on a collapsed model buy nothing |
| a 7B replication | ✋ will not fit a T4 at this batch size. **This is the one weakness a T4 cannot fix** — the paper narrows its claim to 1.5B in the abstract and Threats instead |

### After this runs

1. Paste the LaTeX from cell 3 into `main.tex` as a second results table.
2. Add the CIs from cell 2 to the decomposition table.
3. If WN11's share is near 94 %, change "on YAGO3-10" → "on both graphs" in the
   abstract and conclusion.
4. If it is **not** near 94 %, that is also publishable — report both and say the
   share is graph-dependent. Do not hide it.